# 01 - SVM (classical machine learning)
A Support Vector Machine with an RBF kernel, trained on the raw pixels.

## 1. Imports and settings

In [21]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from torchvision import datasets

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

## 2. Load the data
Download MNIST (first run only) and scale the pixels to 0-1.

In [22]:
train = datasets.MNIST(ROOT / "data", train=True, download=True)
test = datasets.MNIST(ROOT / "data", train=False, download=True)
x_all, y_all = train.data.numpy().astype(np.float32) / 255, train.targets.numpy()
x_test, y_test = test.data.numpy().astype(np.float32) / 255, test.targets.numpy()

## 3. Split the data
54,000 images for training and 6,000 for validation (to compare settings). The 10,000 test images are kept for the end.

In [23]:
x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=6000, stratify=y_all, random_state=SEED)

## 4. Flatten the images
The SVM reads each 28x28 image as a row of 784 numbers.

In [24]:
x_train, x_val, x_test = (a.reshape(len(a), -1) for a in (x_train, x_val, x_test))

## 5. Find good settings
The SVM is slow, so try a few values of C and gamma on 10,000 training images and keep the best one on the validation data.

In [25]:
small = np.random.RandomState(SEED).choice(len(x_train), 10000, replace=False)
results = []
for C in [1, 5, 10]:
    for gamma in [0.01, 0.02, 0.04]:
        val_acc = SVC(kernel="rbf", C=C, gamma=gamma).fit(x_train[small], y_train[small]).score(x_val, y_val)
        results.append((val_acc, C, gamma))
        print(f"C={C}  gamma={gamma}  validation accuracy {val_acc:.4f}")
best_acc, best_C, best_gamma = max(results)
print("best:", best_C, best_gamma)

C=1  gamma=0.01  validation accuracy 0.9578
C=1  gamma=0.02  validation accuracy 0.9650
C=1  gamma=0.04  validation accuracy 0.9657
C=5  gamma=0.01  validation accuracy 0.9672
C=5  gamma=0.02  validation accuracy 0.9723
C=5  gamma=0.04  validation accuracy 0.9692
C=10  gamma=0.01  validation accuracy 0.9678
C=10  gamma=0.02  validation accuracy 0.9722
C=10  gamma=0.04  validation accuracy 0.9692
best: 5 0.02


## 6. Train the final model
Use all 54,000 training images and the best settings.

In [26]:
start = time.time()
model = SVC(kernel="rbf", C=best_C, gamma=best_gamma).fit(x_train, y_train)
train_time = time.time() - start
print(f"training time: {train_time:.0f} s")

training time: 95 s


## 7. Check the accuracy
Test accuracy is the main result. Train accuracy shows how much the model overfits.

In [27]:
test_acc = model.score(x_test, y_test)
train_acc = model.score(x_train[:10000], y_train[:10000])
print(f"test accuracy {test_acc:.4f}   train accuracy {train_acc:.4f}")

test accuracy 0.9851   train accuracy 0.9999


## 8. Save the model
The web app loads this file.

In [28]:
joblib.dump(model, ROOT / "app" / "models" / "classical_svm.joblib", compress=3)

['/Users/seifebeid/Projects/minist/app/models/classical_svm.joblib']

## 9. Save the results
The comparison notebook reads this file. For an SVM, "parameters" is the number of stored support-vector values.

In [29]:
results = dict(test_acc=test_acc, train_acc=train_acc, train_time=train_time,
               parameters=int(model.support_vectors_.size), settings=dict(C=best_C, gamma=best_gamma))
(ROOT / "artifacts" / "svm_results.json").write_text(json.dumps(results, indent=2))

158